# ਪਾਠ 05 - ਏਜੈਂਟਿਕ RAG


## ਸੈਟਅੱਪ

ਇਹ ਨੋਟਬੁੱਕ ਮਾਈਕ੍ਰੋਸਾਫਟ ਏਜੰਟ ਫ੍ਰੇਮਵਰਕ ਦੀ ਵਰਤੋਂ ਕਰਦਿਆਂ ਏਜੰਟਿਕ RAG (ਰੀਟਰੀਵਲ-ਆਗਮੈਂਟਡ ਜੈਨਰੇਸ਼ਨ) ਪੈਟਰਨ ਨੂੰ ਦਰਸਾਉਂਦਾ ਹੈ।

**ਜ਼ਰੂਰੀ ਸ਼ਰਤਾਂ:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — ਤੁਹਾਡਾ Azure AI ਖੋਜ ਸੇਵਾ ਐਂਡਪੋਇੰਟ
- `AZURE_SEARCH_API_KEY` — ਤੁਹਾਡਾ Azure AI ਖੋਜ API ਕੁੰਜੀ
- ਵਾਤਾਵਰਨ ਚਲਾਂਵਿਆਂ ਰਾਹੀਂ Azure OpenAI ਡਿਪਲੋਇਮੈਂਟ ਸੰਰਚਿਤ
- Azure CLI ਪ੍ਰਮਾਣਤ ਕੀਤਾ ਹੋਇਆ (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Agentic RAG ਕੀ ਹੈ?

ਪਾਰੰਪਰਿਕ RAG ਇੱਕ ਨਿਸ਼ਚਿਤ ਪਾਈਪਲਾਈਨ ਦੀ ਪਾਲਣਾ ਕਰਦਾ ਹੈ: ਦਸਤਾਵੇਜ਼ ਪ੍ਰਾਪਤ ਕਰੋ, ਫਿਰ ਜਵਾਬ ਤਿਆਰ ਕਰੋ। **Agentic RAG** ਇਸ ਤੋਂ ਅੱਗੇ ਵੱਧਦਾ ਹੈ ਜਿੱਥੇ ਏਜੰਟ ਨੂੰ ਸਵੈ-ਨਿਯੰਤ੍ਰਣ ਮਿਲਦਾ ਹੈ ਕਿ ਉਹ **ਕਦੋਂ** ਅਤੇ **ਕਿਵੇਂ** ਜਾਣਕਾਰੀ ਪ੍ਰਾਪਤ ਕਰੇ।

Agentic RAG ਨਾਲ, ਏਜੰਟ ਇਹ ਕਰ ਸਕਦਾ ਹੈ:
- ਸਵਾਲ ਦਾ ਜਵਾਬ ਦੇਣ ਤੋਂ ਪਹਿਲਾਂ ਇਹ **ਫੈਸਲਾ** ਕਰਨਾ ਕਿ ਪ੍ਰਾਪਤੀ ਦੀ ਲੋੜ ਹੈ ਜਾਂ ਨਹੀਂ
- ਕਿਹੜਾ ਡਾਟਾ ਸ੍ਰੋਤ ਜਾਂ ਔਜ਼ਾਰ ਪ੍ਰਸ਼ਨ ਕਰਨ ਲਈ **ਚੁਣਨਾ**
- ਪ੍ਰਾਪਤ ਨਤੀਜਿਆਂ ਦਾ **ਮੂਲਯਾਂਕਣ** ਕਰਨਾ ਅਤੇ ਜੇ ਪਹਿਲੀ ਕੋਸ਼ਿਸ਼ ਪੂਰੀ ਨਹੀਂ ਹੋਈ ਤਾਂ ਫਾਲੋ-ਅੱਪ ਪ੍ਰਾਪਤੀ ਕਰਨੀ
- ਕਈ ਪ੍ਰਾਪਤੀ ਸਟੈਪਾਂ ਤੋਂ ਜਾਣਕਾਰੀ ਨੂੰ ਜੋੜ ਕੇ ਇੱਕ ਸਮਨ੍ਵਿਤ ਜਵਾਬ ਤਿਆਰ ਕਰਨਾ

ਇਸ ਨਾਲ ਏਜੰਟ ਸਥਿਰ retrieve-then-generate ਪਾਈਪਲਾਈਨ ਨਾਲੋਂ ਵਧੇਰੇ ਲਚਕੀਲਾ ਅਤੇ ਸਹੀ ਬਣਦਾ ਹੈ।


## ਇੱਕ ਖੋਜ ਟੂਲ ਬਣਾਉਣਾ

Agentic RAG ਵਿੱਚ, ਬਾਹਰੀ ਡੇਟਾ ਸਰੋਤਾਂ ਨੂੰ **ਟੂਲਾਂ** ਵਜੋਂ ਲਪੇਟਿਆ ਜਾਂਦਾ ਹੈ ਜੋ ਏਜੰਟ ਮੰਗ 'ਤੇ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ। ਇਸ ਨਾਲ ਏਜੰਟ ਨੂੰ ਖੋਜ ਨੂੰ ਸਿਰਫ਼ ਇੱਕ ਹੋਰ ਕਾਰਵਾਈ ਵਜੋਂ ਲੈਣ ਦਾ ਮੌਕਾ ਮਿਲਦਾ ਹੈ, ਨਾ ਕਿ ਇੱਕ ਜ਼ਰੂਰੀ ਕਦਮ ਵਜੋਂ।

ਹੇਠਾਂ ਅਸੀਂ ਇੱਕ ਯਾਤਰਾ ਗਿਆਨ ਆਧਾਰ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦੇ ਹਾਂ ਅਤੇ ਇਸਨੂੰ ਇੱਕ ਟੂਲ ਵਜੋਂ ਉਦਘਾਟਨ ਕਰਦੇ ਹਾਂ ਜੋ ਏਜੰਟ ਮੰਜਿਲ ਜਾਣਕਾਰੀ ਲਈ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG ਏਜੰਟ ਬਣਾਉਣਾ

ਹੁਣ ਅਸੀਂ ਇੱਕ ਏਜੰਟ ਬਣਾਉਂਦੇ ਹਾਂ ਜਿਸਨੂੰ **ਜਵਾਬ ਦੇਣ ਤੋਂ ਪਹਿਲਾਂ ਸਦਾ ਜਾਣਕਾਰੀ ਪ੍ਰਾਪਤ ਕਰਨ ਦਾ ਹੁਕਮ ਦਿੱਤਾ ਗਿਆ ਹੈ**। ਏਜੰਟ ਆਪਣੇ ਜਵਾਬਾਂ ਨੂੰ ਆਪਣੇ ਤਿਆਰੀ ਡਾਟਾ 'ਤੇ ਨਿਰਭਰ ਕਰਨ ਦੀ ਬਜਾਏ ਗਿਆਨ ਅਧਾਰ ਵਿੱਚ ਮਜ਼ਬੂਤ ਕਰਨ ਲਈ `search_travel_knowledge` ਟੂਲ ਦੀ ਵਰਤੋਂ ਕਰਦਾ ਹੈ।


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## ਇਟਰੇਟਿਵ ਰੀਟ੍ਰੀਵਲ — ਦ ਮੈਕਰ-ਚੈੱਕਰ ਪੈਟਰਨ

ਏਜੈਂਟਿਕ RAG ਦਾ ਇੱਕ ਮੁੱਖ ਫਾਇਦਾ ਹੈ **ਇਟਰੇਟਿਵ ਰੀਟ੍ਰੀਵਲ**। ਏਜੈਂਟ ਆਪਣੇ ਪਹਿਲਾਂ ਦਿੱਤੇ ਗਿਆ ਨਤੀਜੇ ਦੀ ਪੁਸ਼ਟੀ ਕਰਨ, ਸੁਧਾਰ ਕਰਨ ਜਾਂ ਵਧਾਉਣ ਲਈ ਕਈ ਵਾਰੀ ਖੋਜ ਕਰ ਸਕਦਾ ਹੈ — ਇਸ ਨੂੰ ਇੱਕ "ਮੈਕਰ-ਚੈੱਕਰ" ਵਰਕਫਲੋ ਵਰਗਾ ਸਮਝੋ:

1. **ਮੈਕਰ ਕਦਮ**: ਏਜੈਂਟ ਸ਼ੁਰੂਆਤੀ ਜਾਣਕਾਰੀ ਪ੍ਰਾਪਤ ਕਰਦਾ ਹੈ ਅਤੇ ਜਵਾਬ ਦਾ ਡਰਾਫਟ ਤਿਆਰ ਕਰਦਾ ਹੈ।
2. **ਚੈੱਕਰ ਕਦਮ**: ਏਜੈਂਟ ਤੱਥਾਂ ਦੀ ਪੁਸ਼ਟੀ ਕਰਨ ਜਾਂ ਖਾਮੀਆਂ ਪੂਰੀਆਂ ਕਰਨ ਲਈ ਵਾਧੂ ਖੋਜਾਂ ਕਰਦਾ ਹੈ।

ਹੇਠਾਂ, ਏਜੈਂਟ ਤੋਂ ਇੱਕ ਐਸਾ ਸਵਾਲ ਪੁੱਛਿਆ ਗਿਆ ਹੈ ਜਿਸ ਵਿੱਚ ਕਈ ਮੰਜ਼ਿਲਾਂ ਦੀ ਤੁਲਨਾ ਕਰਨ ਦੀ ਲੋੜ ਹੈ, ਜਿਸ ਕਰਕੇ ਇਹ ਕਈ ਵਾਰੀ ਖੋਜ ਕਰਦਾ ਹੈ।


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Summary

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਸਿੱਖਿਆ ਕਿ ਕਿਵੇਂ ਮਾਈਕ੍ਰੋਸੌਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਇੱਕ **Agentic RAG** ਸਿਸਟਮ ਬਣਾਇਆ ਜਾ ਸਕਦਾ ਹੈ:

- **Agentic RAG** ਏਜੰਟਾਂ ਨੂੰ ਸੁਤੰਤਰ ਹੋਕੇ ਜਾਣਕਾਰੀ ਲੈਣ ਦਾ ਫੈਸਲਾ ਕਰਨ ਦੀ ਆਜ਼ਾਦੀ ਦਿੰਦਾ ਹੈ, ਜਿਸ ਨਾਲ ਜਾਣਕਾਰੀ ਪ੍ਰਾਪਤੀ ਨਿਸ਼ਚਿਤ ਨਹੀਂ ਰਹਿੰਦੀ, ਬਲਕਿ ਗਤੀਸ਼ੀਲ ਬਣ ਜਾਂਦੀ ਹੈ।
- **ਟੂਲਜ਼ ਨੂੰ ਡਾਟਾ ਸਰੋਤ ਵਜੋਂ ਵਰਤਣਾ**: ਬਾਹਰੀ ਗਿਆਨ ਅਧਾਰ (ਜਿਵੇਂ Azure AI Search) ਨੂੰ ਟੂਲਜ਼ ਵਜੋਂ ਪੇਸ਼ ਕੀਤਾ ਜਾਂਦਾ ਹੈ, ਜਿਸ ਨੂੰ ਏਜੰਟ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।
- **ਪੁਨਰਾਵਰਤੀ ਪ੍ਰੀਤੀਵਿਆਰੀ ਐਲਾਨ**: ਮੇਕਰ-ਚੈੱਕਰ ਨਮੂਨਾ ਏਜੰਟ ਨੂੰ ਕਈ ਵਾਰ ਜਾਣਕਾਰੀ ਲੈਣ ਦੇ ਅਵਸਰ ਦਿੰਦਾ ਹੈ — ਖੋਜ, ਪੁਸ਼ਟੀ ਅਤੇ ਸੁਧਾਰ — ਅੰਤ ਵਿੱਚ ਇੱਕ ਆਖਰੀ ਜਵਾਬ ਤਿਆਰ ਕਰਨ ਤੋਂ ਪਹਿਲਾਂ।

ਉਤਪਾਦਨ ਵਿੱਚ, ਤੁਸੀਂ ਮੈਮੋਰੀ `TRAVEL_KNOWLEDGE_BASE` ਨੂੰ ਇੱਕ ਸੱਚੇ Azure AI Search ਇੰਡੈਕਸ ਨਾਲ ਬਦਲਦੇ ਹੋ ਤਾਂ ਜੋ ਵੱਡੇ ਪੱਧਰ ਤੇ ਯਾਤਰਾ ਤੋਂ ਸਬੰਧਿਤ ਦਸਤਾਵੇਜ਼ਾਂ ਨੂੰ ਪ੍ਰਾਪਤ ਕੀਤਾ ਜਾ ਸਕੇ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋكت**:  
ਇਹ ਦਸਤਾਵੇਜ਼ AI ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਅਨੁਵਾਦਿਤ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸ਼ੁੱਧਤਾ ਲਈ ਯਤਨ ਕਰਦੇ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਜਾਣੋ ਕਿ ਆਟੋਮੇਟਿਕ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਹੀਤਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਪ੍ਰਮਾਣਿਕ ਸਗਾ ਵਜੋਂ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੀ ਵਰਤੋਂ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫ਼ਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਸਮਝਾਂ ਲਈ ਜ਼ਿੰਮੇਵਾਰ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
